# Composure

**Composure** measures how well a pitcher maintains their composure in a variety of unfortunate situations.

The score is an aggregation of 14 component statistics, each measuring the rate of an undesirable outcome (walk, ball) following a specific trigger event. All components are normalized within the season and inverted so that **higher = better composure**. Composure+ acts as any other *plus* metric, normalizing player results where a Composure+ of 100 is league average

## Notebook structure

| Phase | What runs | Re-run when... |
|---|---|---|
| **0 — Config** | Thresholds, weights, normalization method | Any parameter change |
| **1 — Data pull** | Fetch pitch-by-pitch from Baseball Savant | New season / first run  |
| **2 — Metric computation** | Compute raw per-pitcher rates from parquet | Changing thresholds or metric definitions |
| **3 — Scoring** | Apply weights + normalization, save results | Changing weights or normalization only |

---
## Metric Definitions

Each metric is a **conditional rate**: given that event X occurred in the previous plate appearance (or the previous pitch), how often does the pitcher produce outcome Y?

---

### After Hard Contact — walks

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `walk_after_barrel` | Previous PA: barrel hit (`launch_speed_angle == 6`) | Next PA: walk or HBP | Did giving up the hardest possible contact cause a loss of command? |
| `walk_after_hard_hit` | Previous PA: exit velo ≥ `HARD_HIT_MPH` (default 90 mph) | Next PA: walk or HBP | Broader hard-contact trigger — any well-struck ball |
| `walk_after_high_xba` | Previous PA: xBa ≥ `HIGH_XBA` (default .450) | Next PA: walk or HBP | Quality-of-contact trigger using expected stats rather than raw velo |

### After Hard Contact — immediate balls

| Metric | Trigger | Outcome | Intuition |
|---|---|---|---|
| `first_ball_after_barrel` | Previous PA: barrel | First pitch of next PA: ball | Quickest possible composure check — did the pitcher come out attacking? |
| `first_ball_after_hard_hit` | Previous PA: exit velo ≥ `HARD_HIT_MPH` | First pitch of next PA: ball | Same idea, broader hard-contact trigger |
| `first_ball_after_high_xba` | Previous PA: xBa ≥ `HIGH_XBA` | First pitch of next PA: ball | Expected-stats version |

---

### After Unlucky Soft Contact — walks
Soft contact is considered *unlucky* based on the expected batting average (xBA) of an event

| Metric | Trigger | Outcome |
|---|---|---
| `walk_after_soft_hit` | Previous PA: exit velo < `SOFT_HIT_MPH` (default 80 mph) | Next PA: walk or HBP |
| `walk_after_low_xba` | Previous PA: xBa < `LOW_XBA` (default .300) | Next PA: walk or HBP |

### After Unlucky Soft Contact — immediate balls

| Metric | Trigger | Outcome |
|---|---|---|
| `first_ball_after_soft_hit` | Previous PA: exit velo < `SOFT_HIT_MPH` | First pitch of next PA: ball |
| `first_ball_after_low_xba` | Previous PA: xBa < `LOW_XBA` | First pitch of next PA: ball |

---

### Additional Metrics

| Metric | Trigger | Outcome |
|---|---|---|
| `hard_hit_after_walk` | Previous PA: walk or HBP | Next PA: exit velo ≥ `HARD_HIT_MPH` |
| `hard_hit_after_soft` | Previous PA: exit velo < `SOFT_HIT_MPH` | Next PA: exit velo ≥ `HARD_HIT_MPH` |
| `walk_after_walk` | Previous PA: walk or HBP | Next PA: walk or HBP |
| `ball_after_foul` | Current PA: foul on pitch N | Pitch N+1 (same PA): ball |

---

### Classification rules
- **Barrel** = `launch_speed_angle == 6` (Based on the statcast classification)
- **Foul tip** → excluded from `ball_after_foul` trigger

---
## Phase 0 — Config & Setup

In [7]:
%pip install pybaseball

Defaulting to user installation because normal site-packages is not writeable
  Using cached pybaseball-2.2.7-py3-none-any.whl.metadata (11 kB)
  Using cached beautifulsoup4-4.14.3-py3-none-any.whl.metadata (3.8 kB)
  Using cached pygithub-2.9.1-py3-none-any.whl.metadata (3.9 kB)
  Using cached soupsieve-2.8.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.5/113.5 kB 4.8 MB/s eta 0:00:00
  Using cached pynacl-1.6.2-cp38-abi3-macosx_10_10_universal2.whl.metadata (10.0 kB)
  Using cached pyjwt-2.12.1-py3-none-any.whl.metadata (4.1 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached pybaseball-2.2.7-py3-none-any.whl (426 kB)
Using cached beautifulsoup4-4.14.3-py3-none-any.whl (107 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 52.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 68.8 MB/s eta 0:00:0000:01

In [16]:
import os, requests
import pandas as pd
import numpy as np
from tqdm import tqdm
import pybaseball
from pybaseball import statcast_single_game, playerid_reverse_lookup
import warnings
warnings.filterwarnings('ignore')
pybaseball.cache.enable()

DATA_DIR = os.path.join(os.path.dirname(os.path.abspath('composure.ipynb')), 'data')
os.makedirs(DATA_DIR, exist_ok=True)

SEASONS = [2023, 2024, 2025]

# ── Metric thresholds (re-run Phase 2 if you change these) ───────────────────
HARD_HIT_MPH  = 90     
SOFT_HIT_MPH  = 80    
HIGH_XBA      = 0.500  
LOW_XBA       = 0.250

# ── Pitcher qualification ─────────────────────────────────────────────────────
MIN_PITCHES = 500      # pitchers below this threshold are excluded from scoring

# ── Per-metric weights (re-run Phase 3 if you change these) ──────────────────
# Default 1.0 = equal contribution. Set to 0 to exclude a metric entirely.

# because walks are more impactful & indicative of composure, they have the highest weights,
# followed by the immediate ball result following an unlucky situation

WEIGHTS = {
    # After hard contact
    'walk_after_barrel':          1.0,
    'walk_after_hard_hit':        1.0,
    'walk_after_high_xba':        1.0,
    'first_ball_after_barrel':    0.5,
    'first_ball_after_hard_hit':  0.5,
    'first_ball_after_high_xba':  0.5,
    # After unlucky soft contact
    'walk_after_soft_hit':        1.0,
    'walk_after_low_xba':         1.0,
    'first_ball_after_soft_hit':  0.5,
    'first_ball_after_low_xba':   0.5,
    # Additional
    'hard_hit_after_walk':        0.75, # hard hits have additional context, should not be weighed as heavily
    'hard_hit_after_soft':        0.75,
    'walk_after_walk':            1.5, # back to back walks are a large indication of composure dropping
    'ball_after_foul':            .05,
}

METRIC_COLS = list(WEIGHTS.keys())

# ── Normalization method (re-run Phase 3 if you change this) ─────────────────
# 'zscore'     — (x - mean) / std across pitchers in the season
# 'percentile' — rank-based 0–1 percentile across pitchers
# 'minmax'     — rescale each metric to [0, 1] across pitchers
NORMALIZATION = 'zscore'

print(f'Config loaded. Data dir: {DATA_DIR}')
print(f'Normalization: {NORMALIZATION} | MIN_PITCHES: {MIN_PITCHES}')
print(f'Thresholds — hard hit: >={HARD_HIT_MPH} mph | soft: <{SOFT_HIT_MPH} mph | '
      f'high xBa: >={HIGH_XBA} | low xBa: <{LOW_XBA}')

Config loaded. Data dir: /Users/aryankapoor/Projects/Baseball/baseball/composure/data
Normalization: zscore | MIN_PITCHES: 500
Thresholds — hard hit: >=90 mph | soft: <80 mph | high xBa: >=0.5 | low xBa: <0.25


---
## Phase 1 — Data Pull

Fetches pitch-by-pitch Statcast data for each season via `pybaseball.statcast_single_game()` and saves the result to `data/all_pitches_YYYY.parquet`.

In [ ]:
# ── Shared fetch helpers ──────────────────────────────────────────────────────

KEEP_COLS = [
    'game_pk', 'game_date', 'pitcher', 'player_name',  # player_name = batter
    'at_bat_number', 'pitch_number',
    'pitch_type', 'pitch_name', 'description', 'events',
    'launch_speed',            
    'launch_speed_angle',        
    'estimated_ba_using_speedangle', 
    'release_speed',
]
NUMERIC_COLS = [
    'launch_speed', 'launch_speed_angle', 'estimated_ba_using_speedangle',
    'at_bat_number', 'pitch_number', 'game_pk', 'release_speed',
]

def _normalize_raw(df):
    df = df[[c for c in KEEP_COLS if c in df.columns]].copy()
    for col in NUMERIC_COLS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df

def _fetch_game(game_pk):
    try:
        df = statcast_single_game(game_pk)
        return _normalize_raw(df) if (df is not None and not df.empty) else None
    except Exception as e:
        print(f'  Error {game_pk}: {e}')
        return None

def _get_game_pks(season):
    """Fetch all completed regular-season gamePks from the MLB Stats API."""
    teams = requests.get('https://statsapi.mlb.com/api/v1/teams?sportId=1').json()['teams']
    pks = set()
    for team in tqdm(teams, desc=f'{season} schedule', leave=False):
        sched = requests.get(
            f'https://statsapi.mlb.com/api/v1/schedule'
            f'?sportId=1&season={season}&teamId={team["id"]}&gameType=R'
        ).json()
        for date in sched.get('dates', []):
            for game in date.get('games', []):
                if game.get('status', {}).get('abstractGameState') == 'Final':
                    pks.add(game['gamePk'])
    return sorted(pks)

def pull_season(season):
    """Load pitch data from parquet cache if available, otherwise fetch and save."""
    path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if os.path.exists(path):
        df = pd.read_parquet(path)
        print(f'{season}: loaded {len(df):,} pitches from cache.')
        return df
    pks = _get_game_pks(season)
    print(f'{season}: fetching {len(pks)} games...')
    results = [_fetch_game(pk) for pk in tqdm(pks, desc=str(season))]
    dfs = [r for r in results if r is not None]
    print(f'  {len(dfs)} ok / {len(pks) - len(dfs)} failed')
    df = pd.concat(dfs, ignore_index=True)
    df.to_parquet(path, index=False)
    print(f'  Saved → {path}')
    return df

print('Fetch helpers defined.')

In [ ]:
# ── 2023 data pull ────────────────────────────────────────────────────────────
pitches_2023 = pull_season(2023)

In [ ]:
# ── 2024 data pull ────────────────────────────────────────────────────────────
pitches_2024 = pull_season(2024)

In [ ]:
# ── 2025 data pull ────────────────────────────────────────────────────────────
pitches_2025 = pull_season(2025)

---
## Phase 2 — Metric Computation

In [9]:
# ── Shared metric helpers ─────────────────────────────────────────────────────

def _add_flags(df):
    d = df.copy()
    # Pitch outcome flags
    d['is_ball']     = d['description'].isin(
                           ['ball', 'ball_in_dirt', 'blocked_ball', 'pitchout', 'intent_ball'])
    d['is_walk']     = d['events'].isin(['walk', 'hit_by_pitch'])  # HBP counts as walk
    d['is_foul']     = d['description'] == 'foul'                  # foul_tip excluded
    # Contact quality flags (NaN launch_speed → False)
    d['is_hard_hit'] = d['launch_speed'] >= HARD_HIT_MPH
    d['is_soft_hit'] = d['launch_speed'] <  SOFT_HIT_MPH
    d['is_barrel_flag'] = (d['launch_speed_angle'] == 6
                           if 'launch_speed_angle' in d.columns else False)
    d['is_high_xba'] = d['estimated_ba_using_speedangle'] >= HIGH_XBA
    d['is_low_xba']  = d['estimated_ba_using_speedangle'] <  LOW_XBA
    # Filled version used for metrics 11 & 12
    d['is_hard_hit_filled'] = d['is_hard_hit'].fillna(False)
    return d

def _pa_summary(pitches):
    p   = pitches.sort_values(['game_pk', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
    grp = ['game_pk', 'pitcher', 'at_bat_number']
    last  = p.groupby(grp).last().reset_index()
    first = (p.groupby(grp)['is_ball'].first()
              .reset_index()
              .rename(columns={'is_ball': 'first_pitch_is_ball'}))
    cols = [
        'game_pk', 'pitcher', 'at_bat_number',
        'is_walk', 'is_hard_hit', 'is_soft_hit', 'is_barrel_flag',
        'is_high_xba', 'is_low_xba', 'is_hard_hit_filled',
    ]
    return last[cols].merge(first, on=grp, how='left')

def _cross_pa_metrics(pa):
    pa = pa.sort_values(['game_pk', 'pitcher', 'at_bat_number']).reset_index(drop=True)
    trigger_cols = ['is_walk', 'is_hard_hit', 'is_soft_hit', 'is_barrel_flag',
                    'is_high_xba', 'is_low_xba']
    prev = (
        pa.groupby(['game_pk', 'pitcher'])[trigger_cols]
          .shift(1)  # shift(1) on ascending at_bat_number = previous PA
          .rename(columns={c: f'prev_{c}' for c in trigger_cols})
    )
    pa = pd.concat([pa, prev], axis=1)

    def rate(trigger, outcome):
        return pa[pa[trigger] == True].groupby('pitcher')[outcome].mean()

    return pd.DataFrame({
        # ── After hard contact ────────────────────────────────────────────
        'walk_after_barrel':          rate('prev_is_barrel_flag', 'is_walk'),
        'walk_after_hard_hit':        rate('prev_is_hard_hit',    'is_walk'),
        'walk_after_high_xba':        rate('prev_is_high_xba',    'is_walk'),
        'first_ball_after_barrel':    rate('prev_is_barrel_flag', 'first_pitch_is_ball'),
        'first_ball_after_hard_hit':  rate('prev_is_hard_hit',    'first_pitch_is_ball'),
        'first_ball_after_high_xba':  rate('prev_is_high_xba',    'first_pitch_is_ball'),
        # ── After unlucky soft contact ────────────────────────────────────
        'walk_after_soft_hit':        rate('prev_is_soft_hit',    'is_walk'),
        'walk_after_low_xba':         rate('prev_is_low_xba',     'is_walk'),
        'first_ball_after_soft_hit':  rate('prev_is_soft_hit',    'first_pitch_is_ball'),
        'first_ball_after_low_xba':   rate('prev_is_low_xba',     'first_pitch_is_ball'),
        # ── Additional ───────────────────────────────────────────────────
        'hard_hit_after_walk':        rate('prev_is_walk',        'is_hard_hit_filled'),
        'hard_hit_after_soft':        rate('prev_is_soft_hit',    'is_hard_hit_filled'),
        'walk_after_walk':            rate('prev_is_walk',        'is_walk'),
    })

def _ball_after_foul(pitches):
    p = pitches.sort_values(['game_pk', 'pitcher', 'at_bat_number', 'pitch_number']).reset_index(drop=True)
    p['prev_is_foul'] = p.groupby(['game_pk', 'pitcher', 'at_bat_number'])['is_foul'].shift(1)
    return (
        p[p['prev_is_foul'] == True]
         .groupby('pitcher')['is_ball']
         .mean()
         .rename('ball_after_foul')
    )

def compute_metrics(season):
    pitches_path = os.path.join(DATA_DIR, f'all_pitches_{season}.parquet')
    if not os.path.exists(pitches_path):
        raise FileNotFoundError(f'Run the Phase 1 pull cell for {season} first.')

    print(f'{season}: loading pitches...')
    pitches = _add_flags(pd.read_parquet(pitches_path))

    print(f'{season}: computing metrics...')
    pa      = _pa_summary(pitches)
    metrics = _cross_pa_metrics(pa).join(_ball_after_foul(pitches), how='outer')

    metrics = metrics.join(pitches.groupby('pitcher').size().rename('pitch_count'))

    out = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    metrics.to_parquet(out)
    print(f'{season}: {len(metrics)} pitchers saved → {out}')
    return metrics

print('Metric helpers defined.')

Metric helpers defined.


In [10]:
# ── 2023 metric computation ───────────────────────────────────────────────────
metrics_2023 = compute_metrics(2023)

2023: loading pitches...
2023: computing metrics...
2023: 863 pitchers saved → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/pitcher_metrics_2023.parquet


In [11]:
# ── 2024 metric computation ───────────────────────────────────────────────────
metrics_2024 = compute_metrics(2024)

2024: loading pitches...
2024: computing metrics...
2024: 852 pitchers saved → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/pitcher_metrics_2024.parquet


In [12]:
# ── 2025 metric computation ───────────────────────────────────────────────────
metrics_2025 = compute_metrics(2025)

2025: loading pitches...
2025: computing metrics...
2025: 872 pitchers saved → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/pitcher_metrics_2025.parquet


---
## Phase 3 — Scoring

Loads `pitcher_metrics_YYYY.parquet` for all seasons, applies the `WEIGHTS` and `NORMALIZATION` method from Phase 0, and produces the final Composure Score and **Composure+**.

### Composure Score formula

For each active metric (weight > 0):
1. Normalize the raw rate across all qualified pitchers in the season using the chosen method
2. Invert the result (×−1) so that lower rate = higher normalized value = better composure
3. Multiply by the metric's weight

Final score = sum of weighted normalized values ÷ sum of weights for metrics with data (NaN metrics don't penalise the pitcher — they simply don't contribute to their average).

### Composure+ formula

A plus stat anchored so that **league average = 100**. Computed from the Composure Score via linear rescaling:

```
composure_plus = 100 + ((composure_score − μ) / σ) × 15
```

where μ and σ are the mean and standard deviation of Composure Score across all qualified pitchers in the season. The ×15 scale factor gives a typical range of roughly 55–145, similar to wRC+ or ERA+. A Composure+ of 115 means the pitcher is one standard deviation above league average composure.

In [17]:
def _normalize(series, method):
    s = series.copy().astype(float)
    if method == 'zscore':
        normed = (s - s.mean()) / s.std()
    elif method == 'percentile':
        normed = s.rank(pct=True, na_option='keep')  
    elif method == 'minmax':
        normed = (s - s.min()) / (s.max() - s.min())
    else:
        raise ValueError(f'Unknown normalization: {method}')
    return normed * -1 


def score_season(season):
    path = os.path.join(DATA_DIR, f'pitcher_metrics_{season}.parquet')
    if not os.path.exists(path):
        raise FileNotFoundError(f'Run the Phase 2 compute cell for {season} first.')

    df     = pd.read_parquet(path)
    df     = df[df['pitch_count'] >= MIN_PITCHES].copy()
    active = [m for m in METRIC_COLS if WEIGHTS.get(m, 0) > 0]

    # Normalize and weight each metric
    normed = pd.DataFrame(index=df.index)
    for col in active:
        normed[col] = _normalize(df[col], NORMALIZATION) * WEIGHTS[col]

    # Weighted mean — denominator uses only metrics with data for that pitcher
    weight_sums = normed.notna().multiply([WEIGHTS[c] for c in active]).sum(axis=1)
    df['composure_score'] = normed.sum(axis=1, skipna=True) / weight_sums

    # Composure+ — league average = 100, std of 15 across qualified pitchers
    # Formula: 100 + ((score - μ) / σ) × 15
    score_mean = df['composure_score'].mean()
    score_std  = df['composure_score'].std()
    df['composure_plus'] = (
        100 + (df['composure_score'] - score_mean) / score_std * 15
    ).round(1)

    # Pitcher names via MLBAM ID reverse lookup
    lookup = playerid_reverse_lookup(df.index.tolist(), key_type='mlbam').set_index('key_mlbam')
    df['pitcher_name'] = df.index.map(lookup['name_last'] + ', ' + lookup['name_first'])
    df['season'] = season

    final = (
        df[['season', 'pitcher_name', 'pitch_count', 'composure_plus', 'composure_score'] + active]
        .dropna(subset=['composure_score'])
        .sort_values('composure_plus', ascending=False)
        .reset_index(drop=True)
    )

    out = os.path.join(DATA_DIR, f'composure_scores_{season}.csv')
    final.to_csv(out, index=False)
    print(f'{season}: {len(final)} qualified pitchers → {out}')
    return final


# ── Score all available seasons ───────────────────────────────────────────────
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_columns', None)

all_scores = {}
for yr in SEASONS:
    if not os.path.exists(os.path.join(DATA_DIR, f'pitcher_metrics_{yr}.parquet')):
        print(f'{yr}: no metrics file yet — run Phase 2 first.')
        continue
    all_scores[yr] = score_season(yr)

for yr, scores in all_scores.items():
    print(f'\n=== {yr} — Top 20 ===')
    display(scores[['pitcher_name', 'pitch_count', 'composure_plus', 'composure_score']].head(20))
    print(f'\n=== {yr} — Bottom 20 ===')
    display(scores[['pitcher_name', 'pitch_count', 'composure_plus', 'composure_score']].tail(20))

2023: 479 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2023.csv
2024: 474 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2024.csv
2025: 479 qualified pitchers → /Users/aryankapoor/Projects/Baseball/baseball/composure/data/composure_scores_2025.csv

=== 2023 — Top 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
0,"martin, chris",736,154.700,1.645
1,"alexander, tyler",679,139.400,1.183
2,"ramirez, nick",659,137.800,1.136
3,"speier, gabe",797,136.600,1.099
4,"nelson, kyle",883,133.300,1.001
5,"kirby, george",2826,131.700,0.954
6,"sánchez, cristopher",1460,128.500,0.858
7,"stephenson, robert",771,128.100,0.844
8,"garcia, robert",530,127.600,0.828
9,"greinke, zack",2221,127.000,0.811



=== 2023 — Bottom 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
459,"abreu, albert",1055,71.800,-0.847
460,"miller, mason",611,71.700,-0.851
461,"saucedo, tayler",791,70.600,-0.885
462,"leclerc, josé",986,70.400,-0.888
463,"yates, kirby",1063,70.200,-0.894
464,"bido, osvaldo",942,70.100,-0.898
465,"marte, yunior",700,69.500,-0.918
466,"kimbrel, craig",1140,69.200,-0.925
467,"fujinami, shintaro",1411,66.200,-1.017
468,"uribe, abner",520,64.600,-1.063



=== 2024 — Top 20 ===


...

,pitcher_name,pitch_count,composure_plus,composure_score
454,"kelly, joe",558,72.700,-0.823
455,"nicolas, kyle",954,71.900,-0.847
456,"buttó, josé",1223,71.300,-0.864
457,"miller, erik",1271,70.700,-0.882
458,"rainey, tanner",943,70.500,-0.889
459,"joyce, ben",541,70.000,-0.903
460,"thorpe, drew",721,69.600,-0.916
461,"diekman, jake",627,69.300,-0.924
462,"medina, luis",724,69.000,-0.932
463,"miller, bobby",1054,68.900,-0.935



=== 2025 — Top 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
0,"dodd, dylan",536,144.200,1.377
1,"martin, chris",651,137.400,1.167
2,"rogers, tyler",978,135.500,1.107
3,"kranick, max",529,132.000,0.998
4,"morejón, adrián",1007,131.800,0.992
5,"brogdon, connor",853,130.600,0.954
6,"walter, brandon",815,130.100,0.936
7,"lee, dylan",1104,128.400,0.884
8,"king, bryan",1046,128.200,0.880
9,"kelly, zack",587,126.100,0.813



=== 2025 — Bottom 20 ===


,pitcher_name,pitch_count,composure_plus,composure_score
459,"pressly, ryan",661,72.200,-0.868
460,"cruz, fernando",793,72.100,-0.868
461,"gonsolin, tony",609,69.900,-0.938
462,"arrighetti, spencer",619,69.800,-0.942
463,"booser, cam",572,69.500,-0.950
464,"fermin, josé",655,69.300,-0.958
465,"zeferjahn, ryan",1039,69.200,-0.960
466,"birdsong, hayden",1195,68.400,-0.984
467,"megill, tylor",1234,68.300,-0.987
468,"nicolas, kyle",652,68.200,-0.992


---
## Multi-year view

Loads saved CSVs to compare composure across seasons. No re-computation needed.

In [15]:
csvs = [
    os.path.join(DATA_DIR, f'composure_scores_{yr}.csv')
    for yr in SEASONS
    if os.path.exists(os.path.join(DATA_DIR, f'composure_scores_{yr}.csv'))
]
all_years = pd.concat([pd.read_csv(p) for p in csvs], ignore_index=True)

print('=== Multi-season pitchers (avg Composure+, min 2 seasons) ===')
multi = (
    all_years.groupby('pitcher_name')
    .agg(
        seasons=('season', 'count'),
        avg_composure_plus=('composure_plus', 'mean'),
        avg_composure_score=('composure_score', 'mean'),
    )
    .query('seasons > 1')
    .sort_values('avg_composure_plus', ascending=False)
)
display(multi.head(30))

=== Multi-season pitchers (avg Composure+, min 2 seasons) ===


,seasons,avg_composure_plus,avg_composure_score
pitcher_name,,,
"martin, chris",3,141.300,1.408
"dodd, dylan",2,132.150,1.111
"speier, gabe",2,127.750,0.941
"rogers, tyler",3,126.833,0.921
"lee, dylan",2,126.800,0.925
"kittredge, andrew",2,122.800,0.782
"stripling, ross",2,121.600,0.727
"kirby, george",3,121.133,0.716
"cecconi, slade",2,121.100,0.724
